<a href="https://colab.research.google.com/github/RJRM999/EDA-DS-Projects/blob/main/Travel_AI_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain langchain-core langchain-community langchain-openai langchain-classic duckduckgo-search python-dotenv requests

In [ ]:
import os
import requests

from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
os.environ["WEATHERAPI_KEY"] = getpass("Enter Weather API Key: ")

Enter OpenAI API Key: ··········
Enter Weather API Key: ··········


Define Tools

In [ ]:
! pip install -U ddgs

In [ ]:
@tool
def get_current_weather(destination: str) -> str:
    """Fetch the current weather for a travel destination using WeatherAPI.com."""
    api_key = os.getenv("WEATHERAPI_KEY")
    if not api_key:
        return "WeatherAPI key is missing. Please add WEATHERAPI_KEY to Colab Secrets."

    url = "https://api.weatherapi.com/v1/current.json"
    params = {"key": api_key, "q": destination, "aqi": "no"}

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        data: Dict[str, Any] = response.json()

        location = data.get("location", {})
        current = data.get("current", {})
        condition = current.get("condition", {}).get("text", "Unknown")

        return (
            f"Weather for {location.get('name', destination)}, "
            f"{location.get('country', '')}:\n"
            f"- Condition: {condition}\n"
            f"- Temperature: {current.get('temp_c')}°C\n"
            f"- Feels like: {current.get('feelslike_c')}°C\n"
            f"- Humidity: {current.get('humidity')}%\n"
            f"- Wind: {current.get('wind_kph')} kph\n"
            f"- Last updated: {current.get('last_updated')}"
        )
    except requests.exceptions.RequestException as exc:
        return f"Could not fetch weather for {destination}. Error: {exc}"
    except KeyError:
        return f"Weather response for {destination} was incomplete. Please check the city name."


search_attractions = DuckDuckGoSearchRun(name="search_top_attractions")
search_attractions.description = (
    "Search the web for top tourist attractions, local places to visit, "
    "food areas, cultural sites, and travel tips for a destination."
)

print("✅ Tools defined.")

✅ Tools defined.


Build the Agent

In [ ]:

def build_agent() -> AgentExecutor:
    """Create and return a LangChain tool-calling Travel Assistant agent."""
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("OPENAI_API_KEY is missing. Please add it to Colab Secrets.")

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    tools = [get_current_weather, search_attractions]

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are an intelligent Travel Assistant AI. "
                "For every destination, use the weather tool to get current weather "
                "and use the search tool to find top attractions. "
                "Then produce a concise, practical travel summary with: "
                "1) Weather, 2) Top attractions, 3) Suggested one-day plan, "
                "4) Practical tips. Do not invent facts if tools fail.",
            ),
            ("human", "Destination: {destination}\nUser request: {input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad"),
        ]
    )

    agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)


def ask_travel_assistant(destination: str, user_request: str = "Plan a short trip.") -> str:
    """Invoke the travel assistant for a destination."""
    executor = build_agent()
    result = executor.invoke({"destination": destination, "input": user_request})
    return result["output"]


print("✅ Agent ready.")

✅ Agent ready.


In [ ]:
city = "Kyoto"
query = "What are the best temples to visit and what's the weather like?"

response = ask_travel_assistant(city, query)
print(response)



> Entering new AgentExecutor chain...

Invoking: `get_current_weather` with `{'destination': 'Kyoto'}`


Weather for Kyoto, Japan:
- Condition: Partly Cloudy
- Temperature: 21.2°C
- Feels like: 21.2°C
- Humidity: 78%
- Wind: 5.4 kph
- Last updated: 2026-05-07 21:45
Invoking: `search_top_attractions` with `{'query': 'best temples to visit in Kyoto'}`


Kyoto's Temples & Shrines: 7 Must-Visit Spots! Planning a trip to Kyoto? With over 1,000 temples and shrines, choosing where to go can be overwhelming! This guide highlights 7 essential spots for your Kyoto adventure. Best Temples In Japan. Travel Inspiration Kyoto Temples. Kyoto Japan Temple Visit.Description. A definitive guide to the top 10 must-visit temples in Kyoto, Japan, showcasing serene beauty, rich history, and spiritual tranquillity. With more than a thousand to choose from, you could spend a lifetime in Kyoto exploring temples. Fortunately, with just a few days in the city, you can visit a selection of the very best the cit

Interactive UI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

city_input = widgets.Text(
    placeholder="e.g. Tokyo, Paris, New York",
    description="Destination:",
    layout=widgets.Layout(width="400px"),
)

query_input = widgets.Textarea(
    value="Plan a short trip.",
    description="Request:",
    layout=widgets.Layout(width="400px", height="80px"),
)

run_button = widgets.Button(
    description="✈️  Get Travel Plan",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)

output_area = widgets.Output()


def on_run_clicked(b):
    city = city_input.value.strip()
    query = query_input.value.strip() or "Plan a short trip."
    if not city:
        with output_area:
            clear_output()
            print("⚠️ Please enter a destination city.")
        return

    with output_area:
        clear_output()
        display(Markdown(f"### ✈️ Planning your trip to **{city}**…"))

    response = ask_travel_assistant(city, query)

    with output_area:
        clear_output()
        display(Markdown(f"## 🗺️ Travel Plan for {city}\n\n{response}"))


run_button.on_click(on_run_clicked)

display(
    widgets.VBox([
        widgets.HTML("<h3>✈️ Travel Assistant</h3>"),
        city_input,
        query_input,
        run_button,
        output_area,
    ])
)



> Entering new AgentExecutor chain...

Invoking: `get_current_weather` with `{'destination': 'Randolph'}`


Weather for Randolph, United States of America:
- Condition: Partly cloudy
- Temperature: 14.4°C
- Feels like: 13.2°C
- Humidity: 60%
- Wind: 16.6 kph
- Last updated: 2026-05-07 12:00
Invoking: `search_top_attractions` with `{'query': 'Randolph'}`


March 4, 2026 - Randolph is a township in southwestern Morris County, in the U.S. state of New Jersey. As of the 2020 United States census, the township's population was 26,504, an increase of 770 (+3.0%) from the 2010 census count of 25,734, which in turn reflected an increase of 887 (+3.6%) from the 24,847 ... March 21, 2026 - Randolph is a suburban city in Norfolk County, Massachusetts, United States. At the 2020 census, the city population was 34,984. Randolph adopted a charter effective January 2010 providing for a council-manager form of government instead of ... 3 weeks ago - Randolph Air Force Base (IATA: RND, ICAO: KRND, FA